# Architectural Tradeoff Matrix

## Prompting vs. RAG vs. SFT vs. Continued Pre-Training

To select the correct architecture for a given task, an LLM must be viewed through three distinct operational components:

* **Parametric Memory:** Weights learned during training
* **Non-Parametric Memory:** External context injected at inference
* **Behavioral Alignment:** The functional format, style, or execution logic applied to inputs

## System Taxonomy & Mechanics

### Prompting (In-Context Learning - ICL)

**Mechanism:** Modifies neither parameters nor external state. Relies entirely on the self-attention mechanism processing $N$ tokens in the context window.

**Primary Drivers:**
* Fast iteration
* Zero infrastructure overhead
* Zero training cost

**System Failure Modes:**
* KV-cache VRAM usage grows quadratically or linearly with context length depending on attention variant (MHA/GQA)
* High Time-To-First-Token (TTFT) latency due to context prefill compute
* Attention distraction (e.g., lost in the middle) degrades retrieval accuracy as token length increases

---

### Retrieval-Augmented Generation (RAG)

**Mechanism:** Couples a frozen LLM (parametric memory) with an external vector, graph, or full-text retrieval system (non-parametric memory).

**Primary Drivers:**
* Injecting dynamic, frequently updated enterprise facts
* Providing verifiable attribution/citations
* Minimizing parametric hallucinations

**System Failure Modes:**
* Retrieval noise injection (irrelevant chunks degrading context window density)
* Multi-hop reasoning failures across fragmented passages
* Added pipeline latency (dense embedding generation + vector index ANN search + reranking + LLM prefill)

---

### Supervised Fine-Tuning (SFT)

**Mechanism:** Updates parametric weights $\theta$ by minimizing cross-entropy loss over a targeted domain or structured dataset.

**Primary Drivers:**
* Behavioral adaptation
* Instruction following
* Enforcing deterministic output schemas (JSON, Cypher, SQL)
* Teaching specialized tools/APIs
* Altering tone/style

**System Failure Modes:**
* **Parametric Memory Fallacy:** Attempting to use SFT to teach vast factual knowledge often leads to high hallucination rates when inputs fall outside the precise training distribution
* Catastrophic forgetting of general reasoning capabilities if data quality and regularization are unmanaged

---

### Continued Pre-Training (Domain-Adaptive Pre-Training - DAPT)

**Mechanism:** Unsupervised casual language modeling ($\text{CausalLM}$) over billions of domain-specific tokens (e.g., raw legal corpora, proprietary code repositories, medical journals) using standard next-token prediction.

**Primary Drivers:**
* Expanding vocabulary/tokenizer density for niche domains
* Shifting the base model's internal prior distributions to match domain jargon, syntax, and implicit semantic relationships

**System Failure Modes:**
* Prohibitively expensive compute requirements
* Severe risk of catastrophically destabilizing original instruction alignment
* Requires subsequent SFT/RL pass before serving

## First-Principles Decision Engine

Use this logical flow when engineering an enterprise ML pipeline:

```
                        [Start: Business Requirements]
                                      │
              Is the goal to inject new/frequently changing 
                  facts or provide strict source citations?
                                ┌─────┴─────┐
                              YES           NO
                               │             │
                         [Deploy RAG]   Is the primary issue 
                                        teaching a NEW format, 
                                        structured output, or 
                                        specialized behavior?
                                             ┌─────┴─────┐
                                           YES           NO
                                            │             │
                                      [Deploy SFT]   Does the domain have 
                                                     a vast, unique vocabulary 
                                                     (e.g., Genomics, Native Code)?
                                                          ┌─────┴─────┐
                                                        YES           NO
                                                         │             │
                                                   [Deploy DAPT]  [Deploy Prompting / ICL]
```

### Decision Logic

1. **RAG Path:** Choose when the task requires injecting new/frequently changing facts or providing strict source citations
2. **SFT Path:** Choose when teaching a NEW format, structured output, or specialized behavior
3. **DAPT Path:** Choose when the domain has a vast, unique vocabulary (e.g., Genomics, Native Code)
4. **Prompting/ICL Path:** Default choice for general-purpose tasks without specialized requirements

## Data Formatting, Tokenization Control, and Loss Masking Mechanics

When fine-tuning base LLMs into instruction-following or chat-aligned models, how data is formatted, tokenized, and processed through the loss function directly dictates output quality, training stability, and VRAM efficiency.

### Chat Templates & String Serialization

Base language models are raw next-token predictors trained on unstructured text sequences. To make them act as assistants, structured interactions (system prompts, user turns, tool calls, assistant turns) must be serialized into a single flat string using explicit control tokens.

Modern architectures use chat templates (typically standardized via Jinja2 templates inside Hugging Face tokenizers) to enforce deterministic boundary markers:

* **Header Control Tokens:** Define who is speaking (`<|start_header_id|>user<|end_header_id|>`, `<|im_start|>assistant`)
* **Turn Termination Tokens:** Signal when a speaker finishes (`<|eot_id|>`, `<|im_end|>`)
* **EOS (End-of-Sequence) Token:** Tells the model the entire generation session is over (`<|endoftext|>`)

**Structural Hazard: Template Mismatch**

If a model is pre-trained with standard ChatML (`<|im_start|>user\n...<|im_end|>`) but fine-tuned with Llama-3 format (`<|start_header_id|>user<|end_header_id|>`), the model must unlearn its embedded attention priors for turn boundaries. This causes severe instability, context leaks, and generation truncation errors.

---

### Loss Masking: Train on Assistant Completions Only

The fundamental rule of Supervised Fine-Tuning (SFT) is: **Only compute gradients and backpropagate loss on tokens the model is expected to generate (the assistant response)**.

**Why Mask Prompts (labels = -100)?**

If you compute cross-entropy loss over the entire sequence (system prompt + user input + assistant response):

* **Model Learns to Predict the User:** The optimizer forces the model parameters to memorize the specific syntax, phrasing, and structure of user queries
* **Gradient Noise & Degradation:** Up to 80% of sequence tokens might be prompt tokens. Computing gradients on prompt tokens causes the model to penalize itself for not "predicting" a user query it could not logically anticipate

**The Mechanics of Cross-Entropy Loss Masking**

In PyTorch, `torch.nn.CrossEntropyLoss` ignores indices set to `-100` by default (`ignore_index=-100`).

Given an input token ID sequence $X = [x_1, x_2, \dots, x_N]$, we construct a parallel label array $Y$:

$$Y_i = \begin{cases} -100 & \text{if } x_i \text{ is part of System, User, or Template control tags} \\ x_i & \text{if } x_i \text{ is part of the Assistant completion} \end{cases}$$

The scalar cross-entropy loss over a sequence of length $N$ becomes:

$$\mathcal{L}_{SFT} = -\frac{1}{\sum_{i=1}^N \mathbb{I}(Y_i \neq -100)} \sum_{i=1}^N \mathbb{I}(Y_i \neq -100) \log P(x_i \mid x_1, x_2, \dots, x_{i-1}; \theta)$$

Where $\mathbb{I}(\cdot)$ is the indicator function. The denominator normalizes loss strictly by the count of active assistant tokens, preventing sequence length bias.

---

### Context Packing Efficiency vs. Padding Waste

Standard batching pads all sequences in a batch to the length of the longest sequence using padding tokens (`<pad>`).

**Naive Batching (High VRAM Waste):**

```
Batch 0: [User... Assistant... <pad> <pad> <pad> <pad> <pad>]  <- 60% pad tokens
Batch 1: [User.............. Assistant....................]  <- Longest sequence
```

**Failure Modes of Naive Padding:**

* **Wasted VRAM & Compute:** Attention matrices ($\mathcal{O}(N^2)$ or $\mathcal{O}(N)$ for FlashAttention) compute operations over padding tokens that are later masked out
* **Bad Batch Statistics:** High percentage of padding distorts loss scaling and slows down gradient updates per batch

**Packing Mechanics (Multipack / Block Diagonal Attention):**

Instead of padding, context packing concatenates multiple independent conversation samples into a single fixed-length sequence (e.g., matching the maximum context length $L = 4096$).

**Packed Sequence (Zero Pad Waste):**

```
[Doc 1 User + Assist <eot>] [Doc 2 User + Assist <eot>] [Doc 3 User...]
```

**Attention Mask Control in Packed Sequences:**

To prevent Sample 2 from attending to tokens in Sample 1 inside the same packed array, we use Block-Diagonal Attention Masks (supported natively by FlashAttention v2/v3 via `cu_seqlens` cumulative sequence length pointers).

$$\text{Attention}(Q, K, V)_{i,j} = \begin{cases} \text{Softmax}\left(\frac{Q_i K_j^T}{\sqrt{d_k}}\right) V_j & \text{if } i, j \text{ belong to the same document and } j \le i \\ 0 & \text{otherwise} \end{cases}$$

This maximizes Tensor Core utilization (100% real token density) and increases fine-tuning throughput by 2x to 5x.

---

**Review Action:** Once you are comfortable with the systems execution view of these concepts, let me know if you are ready to move to the next section on Data Integrity at Scale (N-gram Overlap, Deduplication, and Contamination) or if you want to inspect code.

# Data Pipeline

## Data Integrity at Scale

### N-gram Overlap, Deduplication, and Contamination

Data quality dictates fine-tuning performance. Feeding duplicate samples, overlapping sequences, or contaminated evaluation data into an SFT pipeline causes severe parametric over-fitting, degraded generalization, and distorted metric reporting.

---

### Semantic vs. Near-Duplicate Detection

Duplicates in instruction datasets waste VRAM, artificially inflate training loss convergence metrics, and cause models to output verbatim memorized text rather than learning underlying task logic.

**Exact Deduplication vs. Near-Deduplication**

* **Exact Deduplication:** Simple string/SHA-256 hash collision checks. Fails when minor formatting, trailing whitespace, or template variations exist (e.g., "How do I configure PyTorch?" vs "How do I configure PyTorch?\n")
* **Near-Deduplication (MinHash + Locality-Sensitive Hashing - LSH):** Measures Jaccard Similarity across character/token N-grams without $O(N^2)$ pairwise comparison limits

**The Mathematics of MinHash LSH**

Given two document token sets $A$ and $B$, their Jaccard Similarity is defined as:

$$J(A, B) = \frac{\vert{}A \cap B\vert{}}{\vert{}A \cup B\vert{}}$$

To compute this at scale across millions of training examples:

* **N-gram Extraction:** Convert documents into $k$-shingles (e.g., word 13-grams)
* **Permutation Hashing:** Apply $K$ independent hash functions $\{h_1, h_2, \dots, h_K\}$ to the shingle sets. The minimum hash value across the set for each hash function creates a MinHash Signature vector of length $K$:

$$\text{MinHash}(A)_k = \min_{x \in A} h_k(x)$$

The probability that two MinHash signatures match at index $k$ equals the exact Jaccard similarity:

$$P(\text{MinHash}(A)_k = \text{MinHash}(B)_k) = J(A, B)$$

* **LSH Banding:** Divide the signature vector of size $K$ into $b$ bands of $r$ rows ($K = b \cdot r$). Documents are hashed into candidate buckets per band. Two documents become candidate duplicates if they collide in at least one band

The probability of collision given Jaccard similarity $s$ is:

$$P(\text{Collision}) = 1 - (1 - s^r)^b$$

By tuning $b$ and $r$, we create an S-curve threshold that identifies document pairs exceeding $s \ge 0.8$ similarity in $O(N)$ linear time.

---

### Test Set Contamination Mechanics

Contamination occurs when samples from your validation or benchmark test sets leak into the training dataset.

**Consequences of Contamination**

* **Inverted Evaluation Metrics:** A fine-tuned model scores 95% on a held-out test benchmark, but fails completely in production. It memorized test token sequences instead of learning generalized reasoning
* **Loss Curve Anomaly:** Validation loss decreases rapidly in lockstep with training loss, giving a false signal of generalization

**Detection Protocol: Token N-gram Overlap Detection**

To verify zero contamination between training set $D_{\text{train}}$ and evaluation set $D_{\text{eval}}$:

1. Extract all token $N$-grams (typically $N \in \{8, 13, 50\}$) from each evaluation sample $q \in D_{\text{eval}}$
2. Check for exact matching sub-sequences inside the indexed corpus $D_{\text{train}}$
3. Flag any training sequence where:

$$\text{OverlapRatio}(q, d_{\text{train}}) = \frac{\vert{}\text{Ngrams}(q) \cap \text{Ngrams}(d_{\text{train}})\vert{}}{\vert{}\text{Ngrams}(q)\vert{}} > \tau \quad (\text{typically } \tau \ge 0.2)$$

**De-contamination Action:** Remove the flagged training sample entirely from $D_{\text{train}}$ prior to tokenization and packing.

---

### Sequence Leakage Across Batch Boundaries

Sequence leakage occurs during dataset processing or context packing when sample delimiters are improperly constructed, allowing attention matrices or token sequences to bleed across sample boundaries.

**Leakage Vector 1: Missing Termination Tokens**

If an assistant completion lacks an explicit End-of-Turn (`<|eot_id|>`) or End-of-Sequence (`<|endoftext|>`) token during training:

* The model learns to generate past the response boundary, appending raw template tokens or continuing into the next user prompt during inference

**Leakage Vector 2: Unmasked Cross-Document Attention in Packed Batches**

When packing multiple samples into a single sequence array without Block-Diagonal Attention masks:

* Token $x_i$ in Sample 2 calculates query-key dot products $Q_i K_j^T$ against token $x_j$ in Sample 1
* **Systems Consequence:** The model learns spurious positional dependencies between unrelated conversations, leading to context contamination during multi-turn generation